# Lab 14: The 5 Universal API Components

## Naming what you've already been doing

In labs 9-13 you called Groq's chat endpoint with `call_groq`. This lab gives names to the five parts of that call: endpoint, method, headers, body and response.

Then you put the call into `SimpleAPIClient`, a small class you can reuse. It is not tied to Groq, and it is not tied to chat either.

### Setup - get a free key at https://console.groq.com/keys

In [ ]:
"""Get your free Groq API key at https://console.groq.com/keys - it's never written to this notebook."""
import getpass
import json
import requests

GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"
DEFAULT_MODEL = "llama-3.1-8b-instant"

GROQ_API_KEY = getpass.getpass("Enter your GROQ_API_KEY: ")


def build_request_payload(messages, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The request BODY: what we're asking the model to do."""
    payload = {"model": model, "messages": messages, "temperature": temperature}
    if response_format is not None:
        payload["response_format"] = response_format
    return payload


def extract_reply(response_json):
    """Pulls the assistant's text out of the RESPONSE body."""
    return response_json["choices"][0]["message"]["content"]


def call_groq(messages, api_key=None, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The full request: ENDPOINT + METHOD (POST) + HEADERS (auth) + BODY -> RESPONSE."""
    headers = {
        "Authorization": f"Bearer {api_key or GROQ_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = build_request_payload(messages, model=model, temperature=temperature, response_format=response_format)
    response = requests.post(GROQ_API_URL, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    return extract_reply(response.json())


print("Groq setup ready. Call call_groq(messages) to send a chat completion request.")


## Exercise

### Goal
You write a small reusable "API client" class. An *API* is a web service you talk to by sending requests over the internet and reading the replies. Without a class, you repeat the same `requests.post(...)` code every time. With the class, it remembers the service's address and headers once. Then you call `.post(path, body)` whenever you need it. The same idea is used later to wrap the Groq API.

Some words you need here:
- A *class* is a blueprint. A *method* is a function inside it, like `client.post(...)`. An *attribute* is data stored on the object, like `self.base_url`.
- The *base URL* is the shared start of a web address, like `https://httpbin.org`. The *path* picks the action, like `/post`. Together they make the full URL.
- *Headers* are extra notes sent with a request, as key-value pairs, like `{"X-Test": "1"}`.
- *JSON* is a text format that looks like a Python dict. `requests` can send a dict as JSON with `json=...`. It can read a reply back into a dict with `response.json()`.
- A *POST request* sends data to the server.
- httpbin.org is a free public "echo" service. Whatever you send to `/post` comes back inside its reply, so you can see what you sent.

### What is already written for you
`__init__` is already done. It stores `self.base_url`, with any trailing `/` removed. So `"https://httpbin.org/"` becomes `"https://httpbin.org"`. It also stores `self.headers`, a dict (an empty dict if you pass nothing). You only write `post`.

### `SimpleAPIClient.post(self, path, json_body)`
Code shape (replace each `...`):
```python
def post(self, path, json_body):
    url = ...
    response = requests.post(..., headers=..., json=..., timeout=30)
    response.raise_for_status()
    return ...
```
**You get:**

- `path` (str), starting with `/`, for example `"/post"`
- `json_body` (dict), the data to send, for example `{"hello": "world"}`

**You give back:** a `dict`, the reply body read from JSON.

**Idea:**

1. Put the base URL and the path together to make the full URL: `"https://httpbin.org"` plus `"/post"` gives `"https://httpbin.org/post"`.
2. Send a POST request to that URL with `requests`. Pass `self.headers` as the headers and `json_body` as the JSON body. Also pass a timeout, for example `timeout=30`. Then your program does not wait forever if the server never answers.
3. Call `raise_for_status()` on the response. Every reply has a *status code*, a number: `2xx` (like 200) means success, and `4xx` or `5xx` means an error. `raise_for_status()` does nothing for 2xx. For any other code it raises `requests.exceptions.HTTPError`. So a 404 or 500 becomes an error you see, not a reply that is quietly passed on.
4. Give back `response.json()`.

**Edge cases the solution handles:** a base URL with a trailing slash (`__init__` already handles it), a missing `headers` argument (already handled), and a reply that is not 2xx (it raises `HTTPError`, for example `post("/status/404", {})` on httpbin).

**Example.** This comes from running the reference solution against httpbin:

```python
client = SimpleAPIClient("https://httpbin.org", headers={"X-Test": "1"})
reply = client.post("/post", {"hello": "world"})
reply["json"]                  # {'hello': 'world'}   (the body you sent, echoed back)
reply["headers"]["X-Test"]     # '1'                  (your header, echoed back)
```

**Check:** with the example above, `reply["json"] == {"hello": "world"}` is `True` and `type(reply)` is `dict`. If you call `client.post("/status/404", {})`, it should raise `requests.exceptions.HTTPError`, not give back a value.

### What the checks below test
The cell makes **one real call** to httpbin.org, so you need an internet connection. The asserts check three things:

- `base_url` is `"https://httpbin.org"`.
- The echoed `json` equals `{"hello": "world"}`. This shows you sent the body correctly.
- The echoed header `X-Test` is `"1"`. This shows you sent the headers.

If you see a connection error, check your internet. httpbin is a public service and is sometimes slow. When every assert passes, the cell prints **All checks passed** and reminds you that it made a real network call.

In [ ]:
class SimpleAPIClient:
    def __init__(self, base_url: str, headers: dict | None = None):
        self.base_url = base_url.rstrip("/")
        self.headers = headers or {}

    def post(self, path: str, json_body: dict) -> dict:
        """Send a POST request to base_url + path and give back the JSON reply.

        You get:
            path: the part of the address after the base URL, starting with "/"
                (for example "/post").
            json_body: a dict to send as the JSON body (for example {"hello": "world"}).

        You give back:
            a dict: the reply body read from JSON.

        It raises:
            requests.exceptions.HTTPError: if the server replies with a status
            that is not 2xx.

        Example:
            SimpleAPIClient("https://httpbin.org").post("/post", {"hello": "world"})["json"]
            gives {"hello": "world"} because httpbin sends back what it received.

        TODO:
            1. Join self.base_url and path to make the full URL.
            2. Send the POST with `requests`. Pass self.headers as the headers and
               json_body as the JSON body (add a timeout, such as 30 seconds).
            3. Call raise_for_status() on the response, so an error status raises.
            4. Give back response.json().
        """
        raise NotImplementedError("SimpleAPIClient.post() is not implemented yet - replace the TODO with your code")


client = SimpleAPIClient(base_url="https://httpbin.org", headers={"X-Test": "1"})
assert client.base_url == "https://httpbin.org", 'client.base_url should be ' + repr("https://httpbin.org") + ", got " + repr(client.base_url)
result = client.post("/post", {"hello": "world"})
assert result["json"] == {"hello": "world"}, 'result["json"] should be ' + repr({"hello": "world"}) + ", got " + repr(result["json"])
assert result["headers"]["X-Test"] == "1", 'result["headers"]["X-Test"] should be ' + repr("1") + ", got " + repr(result["headers"]["X-Test"])

print("All checks passed (this made a real call to httpbin.org, a public echo service).")

## Solution

In [ ]:
class SimpleAPIClient:
    def __init__(self, base_url: str, headers: dict | None = None):
        self.base_url = base_url.rstrip("/")
        self.headers = headers or {}

    def post(self, path: str, json_body: dict) -> dict:
        url = f"{self.base_url}{path}"
        response = requests.post(url, headers=self.headers, json=json_body, timeout=30)
        response.raise_for_status()
        return response.json()


print(SimpleAPIClient("https://httpbin.org").post("/post", {"hello": "world"})["json"])

## Real-world project (LIVE - needs your GROQ_API_KEY above)

This repeats lab 9's zero-shot sentiment call. This time it goes through `SimpleAPIClient`, not through the plain `requests.post` inside `call_groq`. You get the same 5 parts and the same result, just wrapped differently.

It reuses `build_zero_shot_messages` and `parse_label` from lab 9. They are written again here, so this notebook stands on its own.

In [ ]:
def build_zero_shot_messages(review: str) -> list[dict]:
    return [
        {
            "role": "system",
            "content": "Classify the sentiment of the review as exactly one word: "
                       "'positive' or 'negative'. Reply with nothing else.",
        },
        {"role": "user", "content": review},
    ]


def parse_label(reply_text: str) -> str:
    normalized = reply_text.strip().strip(".").lower()
    if "negative" in normalized:
        return "negative"
    if "positive" in normalized:
        return "positive"
    return normalized


groq_client = SimpleAPIClient(
    base_url="https://api.groq.com/openai/v1",
    headers={"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"},
)

review = "The battery life on this thing is incredible."
response_body = groq_client.post(
    "/chat/completions",
    {"model": DEFAULT_MODEL, "messages": build_zero_shot_messages(review), "temperature": 0.0},
)
reply_text = response_body["choices"][0]["message"]["content"]
print(f"Review: {review}")
print(f"Predicted sentiment: {parse_label(reply_text)}")